# A04 — host-only DE (C. difficile organoid + butyrate dose-response)

Single donor (patient 35), human GRCh38, polyA mRNA. 18 samples = {Mock,Cd} x {0,5,10 mM} x 3 reps.
First-look: QC -> PCA -> pydeseq2 group contrasts -> A06 gene panels.
Robust 'group' design (infection_butyrate) so contrasts work across pydeseq2 versions.


In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

PROJ = os.path.expanduser('~/Projects_shared/DECIDE_A04_CdiffOrganoid_Butyrate')
TAB  = f'{PROJ}/results/tables'
FIG  = f'{PROJ}/results/figures'
os.makedirs(TAB, exist_ok=True); os.makedirs(FIG, exist_ok=True)
COUNTS = f'{TAB}/counts_raw.tsv'           # from featureCounts
SAMPLES= f'{PROJ}/configs/samples.tsv'


## 1. Load counts + metadata

In [ ]:
# featureCounts: 1 comment line, then Geneid,Chr,Start,End,Strand,Length,<bam...>
cts = pd.read_csv(COUNTS, sep='\t', comment='#')
cts = cts.set_index('Geneid')
count_cols = cts.columns[5:]                # drop Chr..Length
mat = cts[count_cols].copy()
# clean BAM path -> sample_id (strip dir + STAR suffix)
mat.columns = [os.path.basename(c).replace('_Aligned.sortedByCoord.out.bam','') for c in mat.columns]
mat.columns = [c.replace('.bam','') for c in mat.columns]

meta = pd.read_csv(SAMPLES, sep='\t', comment='#')
meta = meta[meta['sample_id'].notna()].set_index('sample_id')
meta['infection']   = meta['infection'].replace({'CDMOI1':'Cd'})
meta['butyrate_mM'] = meta['butyrate_mM'].astype(int).astype(str)
meta['group']       = meta['infection'] + '_' + meta['butyrate_mM']

# align matrix columns to metadata index
common = [s for s in meta.index if s in mat.columns]
missing = [s for s in meta.index if s not in mat.columns]
print('matched samples:', len(common), '| missing from counts:', missing)
mat  = mat[common].T          # samples x genes for pydeseq2
meta = meta.loc[common]
mat.shape, meta['group'].value_counts().to_dict()


## 2. Library sizes + low-count filter

In [ ]:
libsize = mat.sum(axis=1).sort_values()
ax = libsize.plot(kind='barh', figsize=(6,6)); ax.set_xlabel('total counts'); ax.set_title('Library sizes')
plt.tight_layout(); plt.savefig(f'{FIG}/qc_library_sizes.png', dpi=150); plt.show()

keep = (mat.sum(axis=0) >= 10)            # genes with >=10 reads total
mat = mat.loc[:, keep]
print('genes kept:', mat.shape[1])


## 3. DESeq2 (group design) + PCA

In [ ]:
dds = DeseqDataSet(counts=mat, metadata=meta, design_factors='group', refit_cooks=True)
dds.deseq2()

# VST-ish: use normalized log counts for PCA
import numpy as np
norm = pd.DataFrame(dds.layers['normed_counts'], index=mat.index, columns=mat.columns)
logn = np.log2(norm + 1)
# top variable genes
v = logn.var(axis=0).sort_values(ascending=False)
X = logn[v.index[:2000]].values
X = X - X.mean(0)
U,S,Vt = np.linalg.svd(X, full_matrices=False)
pc = U[:, :2] * S[:2]
pcdf = pd.DataFrame(pc, index=mat.index, columns=['PC1','PC2']).join(meta[['infection','butyrate_mM','group']])

fig,ax = plt.subplots(figsize=(6,5))
for g,sub in pcdf.groupby('group'):
    ax.scatter(sub.PC1, sub.PC2, label=g, s=60)
for s,row in pcdf.iterrows(): ax.annotate(s,(row.PC1,row.PC2),fontsize=7,alpha=.6)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend(fontsize=8); ax.set_title('PCA (top 2000 var genes)')
plt.tight_layout(); plt.savefig(f'{FIG}/qc_pca.png', dpi=150); plt.show()


## 4. Contrasts

- infection effect at each butyrate dose: Cd_d vs Mock_d  (d in 0,5,10)
- butyrate dose-response: X_10 vs X_0  (X in Mock, Cd)
The pair of infection effects across doses describes the interaction qualitatively
(does the Cd response shrink as butyrate rises — the A06 hypothesis).

In [ ]:
def run_contrast(dds, a, b, tag):
    st = DeseqStats(dds, contrast=['group', a, b])
    st.summary()
    res = st.results_df.sort_values('padj')
    res.to_csv(f'{TAB}/DE_{tag}.csv')
    n = (res.padj < 0.05).sum()
    print(f'{tag}: {n} genes padj<0.05')
    return res

contrasts = [
    ('Cd_0','Mock_0','infection_at_0mM'),
    ('Cd_5','Mock_5','infection_at_5mM'),
    ('Cd_10','Mock_10','infection_at_10mM'),
    ('Mock_10','Mock_0','butyrate_in_Mock'),
    ('Cd_10','Cd_0','butyrate_in_Cd'),
]
results = {tag: run_contrast(dds, a, b, tag) for a,b,tag in contrasts}


## 5. Volcano per contrast

In [ ]:
def volcano(res, tag, ax):
    x = res['log2FoldChange']; y = -np.log10(res['padj'].clip(lower=1e-300))
    sig = res['padj'] < 0.05
    ax.scatter(x[~sig], y[~sig], s=6, c='lightgray')
    ax.scatter(x[sig],  y[sig],  s=8, c='crimson')
    ax.axhline(-np.log10(0.05), ls='--', c='gray', lw=.7)
    ax.set_title(tag, fontsize=9); ax.set_xlabel('log2FC'); ax.set_ylabel('-log10 padj')

fig,axes = plt.subplots(1,len(results),figsize=(4*len(results),4))
for ax,(tag,res) in zip(np.atleast_1d(axes), results.items()): volcano(res,tag,ax)
plt.tight_layout(); plt.savefig(f'{FIG}/volcano_all.png', dpi=150); plt.show()


## 6. A06 gene panels across the 6 groups

Carry over the A06 readouts so the two models are directly comparable. NOTE: rows use
gene SYMBOLS; if the count matrix is Ensembl IDs, map first (gencode GTF gene_name).

In [ ]:
panels = {
 'ER_chaperones (non-canonical UPR)': ['HSPA5','CALR','WFS1','HYOU1','MANF','PDIA4'],
 'canonical UPR sensors (expect flat)': ['XBP1','ATF6','ATF4','EIF2AK3'],
 'candidate regulators': ['AR','TMPRSS2','BHLHA15'],
 'NF-kB / chemokines': ['NFKB1','RELA','RELB','IRAK2','IKBKE','CXCL1','CXCL2','CXCL3','CXCL8'],
}
# mean log-normalized expression per group
grp_mean = logn.join(meta['group']).groupby('group').mean().T  # genes x groups
order = ['Mock_0','Mock_5','Mock_10','Cd_0','Cd_5','Cd_10']
order = [g for g in order if g in grp_mean.columns]

import matplotlib.pyplot as plt
genes = [g for p in panels.values() for g in p if g in grp_mean.index]
sub = grp_mean.loc[genes, order]
# z-score per gene for display
z = sub.sub(sub.mean(1),axis=0).div(sub.std(1)+1e-9,axis=0)
fig,ax = plt.subplots(figsize=(1.1*len(order)+2, 0.32*len(genes)+2))
im = ax.imshow(z.values, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=45, ha='right')
ax.set_yticks(range(len(genes))); ax.set_yticklabels(genes, fontsize=7)
plt.colorbar(im, label='z (per gene)'); ax.set_title('A06 panels across groups')
plt.tight_layout(); plt.savefig(f'{FIG}/a06_panels_heatmap.png', dpi=150); plt.show()

missing = [g for p in panels.values() for g in p if g not in grp_mean.index]
print('panel genes not in matrix (map IDs?):', missing)


## 7. First-look readout
- Are infection DEGs present at 0 mM and do they shrink at 10 mM? (interaction / A06 hypothesis)
- Do ER chaperones rise with butyrate while canonical UPR sensors stay flat? (non-canonical UPR)
- Do CXCL chemokines fall as butyrate rises in the Cd arm? (chemokine attenuation)
Tables in results/tables/DE_*.csv ; figures in results/figures/.
